# 🏛️ Hierarchical Multi-Agent System

A **Supervisor** decomposes the task into an ordered plan (list of `{agent, instruction}` steps). LangGraph executes the steps sequentially, delegating each to the right specialist worker.

```
 User Task
     │
     ▼
  [Supervisor]  ← with_structured_output → Plan([{agent, instruction}, ...])
     │
     ▼  ◄──────────────────────────────────────────────────┐
  [Worker]  ← create_agent executes current step           │ loop
     │                                                      │
     └── more steps? ──yes────────────────────────────────-─┘
                  └──no──► END
```

**Strengths:** Structured, reproducible, auditable step-by-step.  
**Limitations:** The supervisor plans once up-front — workers cannot trigger mid-plan replanning.

---
```bash
uv sync --group notebooks
```

## ⚙️ Setup

In [1]:
from typing import TypedDict
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from pydantic import BaseModel

load_dotenv()
llm = ChatOpenAI(model="gpt-4o", temperature=0)
print("✅ Setup complete.")

✅ Setup complete.


## 🛠️ Tools & Workers

Three specialist workers, each compiled with `create_agent` and its own tool set. Workers are fully stateless — they receive an instruction and return a result.

In [2]:
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"[Results: {query}]"

@tool
def run_python(code: str) -> str:
    """Execute Python code."""
    return f"[Output: {code}]"

@tool
def write_file(name: str, text: str) -> str:
    """Write text to a file."""
    return f"[Saved: {name}]"


# compile_agent is a thin wrapper around create_agent
def compile_agent(system_prompt: str, tools: list):
    return create_agent(model=llm, tools=tools, system_prompt=system_prompt)

researcher = compile_agent("You are a Researcher. Find information.", [search_web])
coder      = compile_agent("You are a Coder. Write and run Python code.", [run_python])
writer     = compile_agent("You are a Writer. Produce documents.", [write_file])

WORKERS = {"researcher": researcher, "coder": coder, "writer": writer}
print("✅ Workers compiled:", list(WORKERS.keys()))

✅ Workers compiled: ['researcher', 'coder', 'writer']


## 🧠 Supervisor: Structured Planning

Using `llm.with_structured_output` with a Pydantic schema gives us a validated `Plan` object directly — no `JsonOutputParser`, no try/except JSON parsing.

In [3]:
class Step(BaseModel):
    """A single step in the execution plan."""
    agent: str        # one of: researcher, coder, writer
    instruction: str  # the refined sub-task for that agent

class Plan(BaseModel):
    """The supervisor's complete execution plan."""
    steps: list[Step]

# Bind structured output to the LLM
supervisor_llm = llm.with_structured_output(Plan)

SUPERVISOR_SYSTEM = (
    "You are a supervisor. Decompose the task into an ordered list of steps.\n"
    "Assign each step to one of: researcher, coder, writer.\n"
    "Be concise — avoid redundant steps."
)


def plan_task(task: str, history: str) -> Plan:
    """Ask the supervisor to produce a structured plan."""
    user_msg = f"Task: {task}\n\nCompleted so far:\n{history or '(nothing yet)'}"
    plan_raw = supervisor_llm.invoke([
        {"role": "system", "content": SUPERVISOR_SYSTEM},
        {"role": "user",   "content": user_msg},
    ])
    return Plan.model_validate(plan_raw)


# Quick sanity check
sample_plan = plan_task("Research neural networks and write a summary.", "")
for i, step in enumerate(sample_plan.steps, 1):
    print(f"  Step {i}: [{step.agent}] {step.instruction}")

  Step 1: [researcher] Gather information on the history and development of neural networks, including key milestones and figures.
  Step 2: [researcher] Research the basic structure and functioning of neural networks, including concepts like neurons, layers, and activation functions.
  Step 3: [researcher] Investigate different types of neural networks, such as feedforward, convolutional, and recurrent neural networks, and their applications.
  Step 4: [researcher] Explore current trends and advancements in neural network research, including deep learning and neural network architectures.
  Step 5: [coder] Compile and organize the collected research data into a structured format for easy reference.
  Step 6: [writer] Draft a summary of the history and development of neural networks based on the research data.
  Step 7: [writer] Write a section explaining the basic structure and functioning of neural networks.
  Step 8: [writer] Create a section detailing the different types of neural 

## 📋 LangGraph State

`TypedDict` defines the typed state container passed between all graph nodes.  
Every node receives the **full state** and returns an updated copy — never mutated in place.

In [4]:
class MASState(TypedDict):
    task:         str        # the original user task
    steps:        list       # list[Step] from the Plan
    history:      str        # running log of completed steps
    current_step: int        # pointer into steps list

## 🔁 Graph Nodes

Three pure functions: **supervisor** plans, **worker** executes one step, **router** decides if more steps remain.

In [5]:
def supervisor_node(state: MASState) -> MASState:
    """Call the supervisor, store the plan, reset the step counter."""
    plan = plan_task(state["task"], state["history"])
    return {**state, "steps": plan.steps, "current_step": 0}


def worker_node(state: MASState) -> MASState:
    """Execute the current step with the assigned worker agent."""
    step   = state["steps"][state["current_step"]]
    agent  = WORKERS.get(step.agent, researcher)   # safe fallback

    result = agent.invoke({"messages": [{"role": "user", "content": step.instruction}]})
    output = result["messages"][-1].content

    history = state["history"] + f"\n[{step.agent}]: {output}"
    return {**state, "history": history, "current_step": state["current_step"] + 1}


def should_continue(state: MASState) -> str:
    """Loop back to worker while steps remain; otherwise terminate."""
    return "worker" if state["current_step"] < len(state["steps"]) else END

## 🏗️ Build & Compile the Graph

In [6]:
graph = StateGraph(MASState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("worker",     worker_node)

graph.set_entry_point("supervisor")          # planning always runs first
graph.add_edge("supervisor", "worker")        # then execute step 0
graph.add_conditional_edges("worker", should_continue)  # loop or END

app = graph.compile()
print("✅ Hierarchical graph compiled.")

✅ Hierarchical graph compiled.


## ▶️ Demo

A three-phase task that should exercise all three workers in sequence.

In [7]:
result = app.invoke({
    "task":         "Research transformer architectures, implement a toy model, then write a report.",
    "steps":        [],
    "history":      "",
    "current_step": 0,
})

print("\n=== HIERARCHICAL MAS — EXECUTION LOG ===")
print(result["history"])


=== HIERARCHICAL MAS — EXECUTION LOG ===

[researcher]: Here's a summary of the basics of transformer architectures and their key components:

1. **Basics of Transformer Architectures**:
   - Transformers are a type of neural network architecture introduced in the paper "Attention is All You Need" by Vaswani et al. in 2017.
   - They are designed to handle sequential data and have become the foundation for many state-of-the-art models in natural language processing (NLP).
   - Unlike recurrent neural networks (RNNs), transformers do not require sequential data processing, allowing for more parallelization and efficiency.

2. **Self-Attention**:
   - Self-attention is a mechanism that allows the model to weigh the importance of different words in a sentence when encoding a particular word.
   - It computes a set of attention scores that determine how much focus to place on other parts of the input sequence.
   - This mechanism helps capture long-range dependencies and relationships bet

## 💡 Key Takeaways

- **`llm.with_structured_output(Plan)`** replaces `PROMPT | llm | JsonOutputParser()` — Pydantic validation catches malformed plans at parse time.
- **`create_agent`** workers are stateless runnables; they execute assigned instructions and return a final answer.
- To enable **dynamic replanning**, remove the `set_entry_point` bypass and route back through the supervisor after each worker step.
- LangGraph's conditional edges make the loop/terminate logic explicit and easy to test.